In [1]:
import numpy as np
import pandas as pd

import sklearn as sk
from sklearn import base
from sklearn import calibration
from sklearn import cluster
from sklearn import compose
from sklearn import covariance
from sklearn import cross_decomposition
from sklearn import datasets
from sklearn import decomposition
from sklearn import discriminant_analysis
from sklearn import dummy
from sklearn import ensemble
from sklearn import exceptions
from sklearn import experimental
from sklearn import feature_extraction
from sklearn import feature_selection
from sklearn import gaussian_process
from sklearn import impute
from sklearn import inspection
from sklearn import isotonic
from sklearn import kernel_approximation
from sklearn import kernel_ridge
from sklearn import linear_model
from sklearn import manifold
from sklearn import metrics
from sklearn import mixture
from sklearn import model_selection
from sklearn import multiclass
from sklearn import multioutput
from sklearn import naive_bayes
from sklearn import neighbors
from sklearn import neural_network
from sklearn import pipeline
from sklearn import preprocessing
from sklearn import random_projection
from sklearn import semi_supervised
from sklearn import svm
from sklearn import tree
from sklearn import utils

import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns

## Es. 1
Caricare il dataset, eliminare eventuali attributi inutili (giustificare la scelta), eliminare eventuali istanze duplicate, 
eliminare le istanze che contengono valori nulli, trasformare opportunamente i valori categorici (consiglio: usare Label 
Encoder) e dividere il dataset in train (3/4 del dataset) e test (1/4). Calcolare e valutare le predizioni di un 
RandomForestRegressor. Effettuare alcune considerazioni sui risultati ottenuti, calcolando la metrica RMSE.

In [2]:
df = pd.read_csv('../../data/car_price_dataset.csv')
df

,car_ID,symboling,CarName,fueltype,aspiration,doornumber,carbody,drivewheel,enginelocation,wheelbase,...,enginesize,fuelsystem,boreratio,stroke,compressionratio,horsepower,peakrpm,citympg,highwaympg,price
0,1,3,alfa-romero giulia,gas,std,two,convertible,rwd,front,88.6,...,130,mpfi,3.47,2.68,9.0,111,5000,21,27,13495.0
1,2,3,alfa-romero stelvio,gas,std,two,convertible,rwd,front,88.6,...,130,mpfi,3.47,2.68,9.0,111,5000,21,27,16500.0
2,3,1,alfa-romero Quadrifoglio,gas,std,two,hatchback,rwd,front,94.5,...,152,mpfi,2.68,3.47,9.0,154,5000,19,26,16500.0
3,4,2,audi 100 ls,gas,std,four,sedan,fwd,front,99.8,...,109,mpfi,3.19,3.40,10.0,102,5500,24,30,13950.0
4,5,2,audi 100ls,gas,std,four,sedan,4wd,front,99.4,...,136,mpfi,3.19,3.40,8.0,115,5500,18,22,17450.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
200,201,-1,volvo 145e (sw),gas,std,four,sedan,rwd,front,109.1,...,141,mpfi,3.78,3.15,9.5,114,5400,23,28,16845.0
201,202,-1,volvo 144ea,gas,turbo,four,sedan,rwd,front,109.1,...,141,mpfi,3.78,3.15,8.7,160,5300,19,25,19045.0
202,203,-1,volvo 244dl,gas,std,four,sedan,rwd,front,109.1,...,173,mpfi,3.58,2.87,8.8,134,5500,18,23,21485.0
203,204,-1,volvo 246,diesel,turbo,four,sedan,rwd,front,109.1,...,145,idi,3.01,3.40,23.0,106,4800,26,27,22470.0


- È assolutamente sbagliato mantenere la colonna id, deve essere eliminata per evitare overfitting.
- La colonna CarName è inutile ai fini della predizione

In [3]:
df.drop(['car_ID'], axis=1, inplace=True)
df

,symboling,CarName,fueltype,aspiration,doornumber,carbody,drivewheel,enginelocation,wheelbase,carlength,...,enginesize,fuelsystem,boreratio,stroke,compressionratio,horsepower,peakrpm,citympg,highwaympg,price
0,3,alfa-romero giulia,gas,std,two,convertible,rwd,front,88.6,168.8,...,130,mpfi,3.47,2.68,9.0,111,5000,21,27,13495.0
1,3,alfa-romero stelvio,gas,std,two,convertible,rwd,front,88.6,168.8,...,130,mpfi,3.47,2.68,9.0,111,5000,21,27,16500.0
2,1,alfa-romero Quadrifoglio,gas,std,two,hatchback,rwd,front,94.5,171.2,...,152,mpfi,2.68,3.47,9.0,154,5000,19,26,16500.0
3,2,audi 100 ls,gas,std,four,sedan,fwd,front,99.8,176.6,...,109,mpfi,3.19,3.40,10.0,102,5500,24,30,13950.0
4,2,audi 100ls,gas,std,four,sedan,4wd,front,99.4,176.6,...,136,mpfi,3.19,3.40,8.0,115,5500,18,22,17450.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
200,-1,volvo 145e (sw),gas,std,four,sedan,rwd,front,109.1,188.8,...,141,mpfi,3.78,3.15,9.5,114,5400,23,28,16845.0
201,-1,volvo 144ea,gas,turbo,four,sedan,rwd,front,109.1,188.8,...,141,mpfi,3.78,3.15,8.7,160,5300,19,25,19045.0
202,-1,volvo 244dl,gas,std,four,sedan,rwd,front,109.1,188.8,...,173,mpfi,3.58,2.87,8.8,134,5500,18,23,21485.0
203,-1,volvo 246,diesel,turbo,four,sedan,rwd,front,109.1,188.8,...,145,idi,3.01,3.40,23.0,106,4800,26,27,22470.0


In [4]:
df.duplicated()

0      False
1      False
2      False
3      False
4      False
       ...  
200    False
201    False
202    False
203    False
204    False
Length: 205, dtype: bool

In [5]:
df = df.drop_duplicates()
df

,symboling,CarName,fueltype,aspiration,doornumber,carbody,drivewheel,enginelocation,wheelbase,carlength,...,enginesize,fuelsystem,boreratio,stroke,compressionratio,horsepower,peakrpm,citympg,highwaympg,price
0,3,alfa-romero giulia,gas,std,two,convertible,rwd,front,88.6,168.8,...,130,mpfi,3.47,2.68,9.0,111,5000,21,27,13495.0
1,3,alfa-romero stelvio,gas,std,two,convertible,rwd,front,88.6,168.8,...,130,mpfi,3.47,2.68,9.0,111,5000,21,27,16500.0
2,1,alfa-romero Quadrifoglio,gas,std,two,hatchback,rwd,front,94.5,171.2,...,152,mpfi,2.68,3.47,9.0,154,5000,19,26,16500.0
3,2,audi 100 ls,gas,std,four,sedan,fwd,front,99.8,176.6,...,109,mpfi,3.19,3.40,10.0,102,5500,24,30,13950.0
4,2,audi 100ls,gas,std,four,sedan,4wd,front,99.4,176.6,...,136,mpfi,3.19,3.40,8.0,115,5500,18,22,17450.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
200,-1,volvo 145e (sw),gas,std,four,sedan,rwd,front,109.1,188.8,...,141,mpfi,3.78,3.15,9.5,114,5400,23,28,16845.0
201,-1,volvo 144ea,gas,turbo,four,sedan,rwd,front,109.1,188.8,...,141,mpfi,3.78,3.15,8.7,160,5300,19,25,19045.0
202,-1,volvo 244dl,gas,std,four,sedan,rwd,front,109.1,188.8,...,173,mpfi,3.58,2.87,8.8,134,5500,18,23,21485.0
203,-1,volvo 246,diesel,turbo,four,sedan,rwd,front,109.1,188.8,...,145,idi,3.01,3.40,23.0,106,4800,26,27,22470.0


In [6]:
df.isnull().sum()

symboling           0
CarName             0
fueltype            0
aspiration          0
doornumber          0
carbody             0
drivewheel          0
enginelocation      0
wheelbase           0
carlength           0
carwidth            0
carheight           0
curbweight          0
enginetype          0
cylindernumber      0
enginesize          0
fuelsystem          0
boreratio           0
stroke              0
compressionratio    0
horsepower          0
peakrpm             0
citympg             0
highwaympg          0
price               0
dtype: int64

In [7]:
cat_features = [col for col in df.columns if df[col].dtype == object]
cat_features

['CarName',
 'fueltype',
 'aspiration',
 'doornumber',
 'carbody',
 'drivewheel',
 'enginelocation',
 'enginetype',
 'cylindernumber',
 'fuelsystem']

In [8]:
df.loc[:, cat_features]

,CarName,fueltype,aspiration,doornumber,carbody,drivewheel,enginelocation,enginetype,cylindernumber,fuelsystem
0,alfa-romero giulia,gas,std,two,convertible,rwd,front,dohc,four,mpfi
1,alfa-romero stelvio,gas,std,two,convertible,rwd,front,dohc,four,mpfi
2,alfa-romero Quadrifoglio,gas,std,two,hatchback,rwd,front,ohcv,six,mpfi
3,audi 100 ls,gas,std,four,sedan,fwd,front,ohc,four,mpfi
4,audi 100ls,gas,std,four,sedan,4wd,front,ohc,five,mpfi
...,...,...,...,...,...,...,...,...,...,...
200,volvo 145e (sw),gas,std,four,sedan,rwd,front,ohc,four,mpfi
201,volvo 144ea,gas,turbo,four,sedan,rwd,front,ohc,four,mpfi
202,volvo 244dl,gas,std,four,sedan,rwd,front,ohcv,six,mpfi
203,volvo 246,diesel,turbo,four,sedan,rwd,front,ohc,six,idi


In [9]:
ord_enc = preprocessing.OrdinalEncoder()
ord_enc.fit(df.loc[:, cat_features])

OrdinalEncoder()

In [10]:
enc_features = ord_enc.transform(df.loc[:, cat_features])
enc_features

array([[  2.,   1.,   0., ...,   0.,   2.,   5.],
       [  3.,   1.,   0., ...,   0.,   2.,   5.],
       [  1.,   1.,   0., ...,   5.,   3.,   5.],
       ...,
       [140.,   1.,   0., ...,   5.,   3.,   5.],
       [142.,   0.,   1., ...,   3.,   3.,   3.],
       [143.,   1.,   1., ...,   3.,   2.,   5.]])

In [11]:
df.loc[:, cat_features] = enc_features
df

,symboling,CarName,fueltype,aspiration,doornumber,carbody,drivewheel,enginelocation,wheelbase,carlength,...,enginesize,fuelsystem,boreratio,stroke,compressionratio,horsepower,peakrpm,citympg,highwaympg,price
0,3,2.0,1.0,0.0,1.0,0.0,2.0,0.0,88.6,168.8,...,130,5.0,3.47,2.68,9.0,111,5000,21,27,13495.0
1,3,3.0,1.0,0.0,1.0,0.0,2.0,0.0,88.6,168.8,...,130,5.0,3.47,2.68,9.0,111,5000,21,27,16500.0
2,1,1.0,1.0,0.0,1.0,2.0,2.0,0.0,94.5,171.2,...,152,5.0,2.68,3.47,9.0,154,5000,19,26,16500.0
3,2,4.0,1.0,0.0,0.0,3.0,1.0,0.0,99.8,176.6,...,109,5.0,3.19,3.40,10.0,102,5500,24,30,13950.0
4,2,5.0,1.0,0.0,0.0,3.0,0.0,0.0,99.4,176.6,...,136,5.0,3.19,3.40,8.0,115,5500,18,22,17450.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
200,-1,139.0,1.0,0.0,0.0,3.0,2.0,0.0,109.1,188.8,...,141,5.0,3.78,3.15,9.5,114,5400,23,28,16845.0
201,-1,138.0,1.0,1.0,0.0,3.0,2.0,0.0,109.1,188.8,...,141,5.0,3.78,3.15,8.7,160,5300,19,25,19045.0
202,-1,140.0,1.0,0.0,0.0,3.0,2.0,0.0,109.1,188.8,...,173,5.0,3.58,2.87,8.8,134,5500,18,23,21485.0
203,-1,142.0,0.0,1.0,0.0,3.0,2.0,0.0,109.1,188.8,...,145,3.0,3.01,3.40,23.0,106,4800,26,27,22470.0


In [12]:
X = df.drop(['price'], axis=1)
X

,symboling,CarName,fueltype,aspiration,doornumber,carbody,drivewheel,enginelocation,wheelbase,carlength,...,cylindernumber,enginesize,fuelsystem,boreratio,stroke,compressionratio,horsepower,peakrpm,citympg,highwaympg
0,3,2.0,1.0,0.0,1.0,0.0,2.0,0.0,88.6,168.8,...,2.0,130,5.0,3.47,2.68,9.0,111,5000,21,27
1,3,3.0,1.0,0.0,1.0,0.0,2.0,0.0,88.6,168.8,...,2.0,130,5.0,3.47,2.68,9.0,111,5000,21,27
2,1,1.0,1.0,0.0,1.0,2.0,2.0,0.0,94.5,171.2,...,3.0,152,5.0,2.68,3.47,9.0,154,5000,19,26
3,2,4.0,1.0,0.0,0.0,3.0,1.0,0.0,99.8,176.6,...,2.0,109,5.0,3.19,3.40,10.0,102,5500,24,30
4,2,5.0,1.0,0.0,0.0,3.0,0.0,0.0,99.4,176.6,...,1.0,136,5.0,3.19,3.40,8.0,115,5500,18,22
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
200,-1,139.0,1.0,0.0,0.0,3.0,2.0,0.0,109.1,188.8,...,2.0,141,5.0,3.78,3.15,9.5,114,5400,23,28
201,-1,138.0,1.0,1.0,0.0,3.0,2.0,0.0,109.1,188.8,...,2.0,141,5.0,3.78,3.15,8.7,160,5300,19,25
202,-1,140.0,1.0,0.0,0.0,3.0,2.0,0.0,109.1,188.8,...,3.0,173,5.0,3.58,2.87,8.8,134,5500,18,23
203,-1,142.0,0.0,1.0,0.0,3.0,2.0,0.0,109.1,188.8,...,3.0,145,3.0,3.01,3.40,23.0,106,4800,26,27


In [13]:
y = df['price']
y

0      13495.0
1      16500.0
2      16500.0
3      13950.0
4      17450.0
        ...   
200    16845.0
201    19045.0
202    21485.0
203    22470.0
204    22625.0
Name: price, Length: 205, dtype: float64

In [14]:
X_train, X_test, y_train, y_test = model_selection.train_test_split(X, y, test_size=0.25, random_state=42)

In [15]:
model = ensemble.RandomForestRegressor()
model.fit(X=X_train, y=y_train)

RandomForestRegressor()

In [16]:
y_pred = model.predict(X_train)
print('RMSE on training samples:', metrics.root_mean_squared_error(y_train, y_pred))
y_pred = model.predict(X_test)
print('RMSE on test samples:', metrics.root_mean_squared_error(y_test, y_pred))

RMSE on training samples: 904.5897016041947
RMSE on test samples: 1972.2073028765744


## Es. 2
Creare una pipeline in cui, a partire dal dataset utilizzato al punto precedente, i valori degli attributi carlength, 
carwidth e carheight sono discretizzati in 5 intervalli, citympg e highwaympg sono trasformati con uno 
StandardScaler e tutti gli altri attributi sono lasciati invariati.

In [17]:
coltran = compose.ColumnTransformer(transformers=[("discr", preprocessing.KBinsDiscretizer(n_bins=5), ["carlength", "carwidth", "carheight"]),
                                                  ("std", preprocessing.StandardScaler(), ["citympg", "highwaympg"])],
                                    remainder='passthrough')

`remainder='passthrough'` è molto importante siccome, se non specificato, le feature che non sono interessate da nessuna trasformazione sono scartate, rimosse dal dataset trasformato.

In [18]:
my_pipeline = pipeline.Pipeline(steps=[('coltran', coltran),
                                    ('estimator', ensemble.RandomForestRegressor())])

In [19]:
my_pipeline.fit(X=X_train, y=y_train)


/opt/anaconda3/envs/BDTA/lib/python3.13/site-packages/sklearn/compose/_column_transformer.py:1667: FutureWarning: 
The format of the columns of the 'remainder' transformer in ColumnTransformer.transformers_ will change in version 1.7 to match the format of the other transformers.
At the moment the remainder columns are stored as indices (of type int). With the same ColumnTransformer configuration, in the future they will be stored as column names (of type str).
To use the new behavior now and suppress this warning, use ColumnTransformer(force_int_remainder_cols=False).

  warnings.warn(


Pipeline(steps=[('coltran',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('discr', KBinsDiscretizer(),
                                                  ['carlength', 'carwidth',
                                                   'carheight']),
                                                 ('std', StandardScaler(),
                                                  ['citympg', 'highwaympg'])])),
                ('estimator', RandomForestRegressor())])

In [20]:
y_pred = my_pipeline.predict(X=X_train)
print('RMSE on training samples:', metrics.root_mean_squared_error(y_train, y_pred))
y_pred = my_pipeline.predict(X=X_test)
print('RMSE on test samples:', metrics.root_mean_squared_error(y_test, y_pred))

RMSE on training samples: 878.3482486486124
RMSE on test samples: 1981.493017144881


## Es. 3
Creare una nuova pipeline che applica la SelectKBest al dataset utilizzato al punto 1 e aggiunge le componenti 
ottenute alle componenti della pipeline del punto precedente. Valutare i valori migliori di k di SelectKBest, del numero 
di intervalli in cui discretizzare carlength, carwidth e carheight e dei parametri criterion e max_depth del 
RandomForestRegressor. Ignorare eventuali warning. Confrontare i risultati con quelli ottenuti precedentemente.

In [21]:
combined_features = pipeline.FeatureUnion([('kbest', feature_selection.SelectKBest()), ('coltran', coltran)])

my_pipeline = pipeline.Pipeline(steps=[('combined_features', combined_features),
                                        ('estimator', ensemble.RandomForestRegressor())],
                              #verbose = True
                              )

parameters = {
    'combined_features__kbest__k': [2, 4, 6],
    'combined_features__coltran__discr__n_bins': [5, 10],
    'estimator__criterion': ['squared_error', 'absolute_error', 'friedman_mse', 'poisson'],
    'estimator__max_depth': [7, 10]
}

gd = model_selection.GridSearchCV(my_pipeline, parameters)
gd.fit(X_train, y_train)
y_pred = gd.predict(X_test)
print('RMSE:', metrics.root_mean_squared_error(y_test, y_pred))
gd.best_params_

/opt/anaconda3/envs/BDTA/lib/python3.13/site-packages/sklearn/feature_selection/_univariate_selection.py:112: RuntimeWarning: divide by zero encountered in divide
  f = msb / msw
/opt/anaconda3/envs/BDTA/lib/python3.13/site-packages/sklearn/feature_selection/_univariate_selection.py:112: RuntimeWarning: divide by zero encountered in divide
  f = msb / msw
/opt/anaconda3/envs/BDTA/lib/python3.13/site-packages/sklearn/feature_selection/_univariate_selection.py:112: RuntimeWarning: divide by zero encountered in divide
  f = msb / msw
/opt/anaconda3/envs/BDTA/lib/python3.13/site-packages/sklearn/feature_selection/_univariate_selection.py:112: RuntimeWarning: divide by zero encountered in divide
  f = msb / msw
/opt/anaconda3/envs/BDTA/lib/python3.13/site-packages/sklearn/feature_selection/_univariate_selection.py:112: RuntimeWarning: divide by zero encountered in divide
  f = msb / msw
/opt/anaconda3/envs/BDTA/lib/python3.13/site-packages/sklearn/feature_selection/_univariate_selection.py:

RMSE: 1962.9994389852536


{'combined_features__coltran__discr__n_bins': 5,
 'combined_features__kbest__k': 4,
 'estimator__criterion': 'squared_error',
 'estimator__max_depth': 10}

In [22]:
my_pipeline

Pipeline(steps=[('combined_features',
                 FeatureUnion(transformer_list=[('kbest', SelectKBest()),
                                                ('coltran',
                                                 ColumnTransformer(remainder='passthrough',
                                                                   transformers=[('discr',
                                                                                  KBinsDiscretizer(),
                                                                                  ['carlength',
                                                                                   'carwidth',
                                                                                   'carheight']),
                                                                                 ('std',
                                                                                  StandardScaler(),
                                                                                  ['citympg',
                                                                                   'highwaympg'])]))])),
                ('estimator', RandomForestRegressor())])

## Es. 4
Creare una pipeline che, a partire dal dataset originale, trasforma le colonne testuali in valori numerici e le feature 
numeriche attraverso lo StandardScaler e applica il RandomForestRegressor. Come variano le performance?

In [23]:
df = pd.read_csv('../../data/car_price_dataset.csv')
df

,car_ID,symboling,CarName,fueltype,aspiration,doornumber,carbody,drivewheel,enginelocation,wheelbase,...,enginesize,fuelsystem,boreratio,stroke,compressionratio,horsepower,peakrpm,citympg,highwaympg,price
0,1,3,alfa-romero giulia,gas,std,two,convertible,rwd,front,88.6,...,130,mpfi,3.47,2.68,9.0,111,5000,21,27,13495.0
1,2,3,alfa-romero stelvio,gas,std,two,convertible,rwd,front,88.6,...,130,mpfi,3.47,2.68,9.0,111,5000,21,27,16500.0
2,3,1,alfa-romero Quadrifoglio,gas,std,two,hatchback,rwd,front,94.5,...,152,mpfi,2.68,3.47,9.0,154,5000,19,26,16500.0
3,4,2,audi 100 ls,gas,std,four,sedan,fwd,front,99.8,...,109,mpfi,3.19,3.40,10.0,102,5500,24,30,13950.0
4,5,2,audi 100ls,gas,std,four,sedan,4wd,front,99.4,...,136,mpfi,3.19,3.40,8.0,115,5500,18,22,17450.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
200,201,-1,volvo 145e (sw),gas,std,four,sedan,rwd,front,109.1,...,141,mpfi,3.78,3.15,9.5,114,5400,23,28,16845.0
201,202,-1,volvo 144ea,gas,turbo,four,sedan,rwd,front,109.1,...,141,mpfi,3.78,3.15,8.7,160,5300,19,25,19045.0
202,203,-1,volvo 244dl,gas,std,four,sedan,rwd,front,109.1,...,173,mpfi,3.58,2.87,8.8,134,5500,18,23,21485.0
203,204,-1,volvo 246,diesel,turbo,four,sedan,rwd,front,109.1,...,145,idi,3.01,3.40,23.0,106,4800,26,27,22470.0


In [24]:
X = df.drop(['car_ID', 'price'], axis= 1)
y = df['price']

In [25]:
categorical_features = [col for col in X.columns if df[col].dtype == object]
numeric_features = [col for col in X.columns if col not in categorical_features]
print(categorical_features)
print(numeric_features)

['CarName', 'fueltype', 'aspiration', 'doornumber', 'carbody', 'drivewheel', 'enginelocation', 'enginetype', 'cylindernumber', 'fuelsystem']
['symboling', 'wheelbase', 'carlength', 'carwidth', 'carheight', 'curbweight', 'enginesize', 'boreratio', 'stroke', 'compressionratio', 'horsepower', 'peakrpm', 'citympg', 'highwaympg']


In [26]:
X_train, X_test, y_train, y_test = model_selection.train_test_split(X, y, test_size=0.25, random_state=42)

In [27]:
coltran = compose.ColumnTransformer(transformers=[("onehot", preprocessing.OneHotEncoder(handle_unknown='ignore'), categorical_features),
                                          ("std", preprocessing.StandardScaler(), numeric_features)],
                                 remainder='passthrough')

my_pipeline = pipeline.Pipeline(steps=[("coltran", coltran),
                           ("estimator", ensemble.RandomForestRegressor())])

my_pipeline.fit(X=X_train, y=y_train)

y_pred = my_pipeline.predict(X=X_train)
print('RMSE on training samples:', metrics.root_mean_squared_error(y_train, y_pred))
y_pred = my_pipeline.predict(X=X_test)
print('RMSE on test samples:', metrics.root_mean_squared_error(y_test, y_pred))

RMSE on training samples: 925.2787707198147
RMSE on test samples: 2053.9241776442045
